# Exercise on the use of OLS, Lasso, Ridge and PCR regression

In this exercise we'll check the difference between some regression algorithms.

The goal is to predict a measure of the progression of diabetes from some input features, such as age, body weight, etc.

## The diabetes dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

data = load_diabetes(scaled = False)
features = data.feature_names
X, y = data.data, data.target

print(data.DESCR)

In [ ]:
# Quick look at the data
df = pd.DataFrame(X, columns=features)
df['target'] = y
df.describe().round(2)

We can produce 2D plots to show the marginal relationship between each feature and the target. Be careful though: these simple plots do not account for the effect of other variables and can therfore be misleading if read alone.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=True)

for i, ax in enumerate(axes.flatten()):
    ax.scatter(X[:, i], y, alpha=0.4, s=10)
    ax.set_xlabel(features[i])

axes[0, 0].set_ylabel('Target')
axes[1, 0].set_ylabel('Target')
plt.tight_layout()
plt.show()

## Splitting and scaling the dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# create the test and train datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

# scale and center the data
scalerX = StandardScaler()

X0_train = scalerX.fit_transform(X_train)

In [ ]:
# Look at the data in X0_train now: does it look as expected?
df = pd.DataFrame(X0_train, columns=features)
df['target'] = y_train
df.describe().round(2)

# Ordinary Least-Squares regression

A general regression problem can be written as: $y = f(\mathbf{x})$  with $y \in \mathbb{R}$, $x \in \mathbb{R}^d$ and $f: \mathbb{R}^d \mapsto \mathbb{R}$.
In linear regression, the function is represented by an array of weights: $y = \mathbf{w}^T \mathbf{x} = \sum_{i}^d w_i x_i$.

We need to tune the weights to our process, so we collect some data on the inputs $\mathbf{x}$ and the target $y$. The objective is to tune the weights to minimize the euclidean distance between the observations $\mathbf{y} \in \mathbb{R}^n$ and the predictions $\mathbf{X} \mathbf{w}$:

$\mathbf{w} = \underset{\mathbf{w}}{\mathrm{min}} ||\mathbf{X}\mathbf{w} - \mathbf{y}||^2_2$

In [ ]:
from sklearn.linear_model import LinearRegression

# Create the linear regression object
OLS_reg = LinearRegression().fit(X0_train, y_train)

# To test the regression, we need to scale and center also the test data
X0_test = scalerX.transform(X_test)
y_pred_OLS = OLS_reg.predict(X0_test)

plt.scatter(y_test, y_pred_OLS)
plt.plot(y_test, y_test, c='r', alpha=0.6, ls='--')
plt.xlim(y_test.min()-5, y_test.max()+5)
plt.ylim(y_test.min()-5, y_test.max()+5)
plt.xlabel('Observed target')
plt.ylabel('Predicted target')
plt.show()


## Performance metrics

To assess the quality of the regression model, we need to compare the predictions and observations for some points that were not used to train the model.
There are a huge number of different metrics that can be used depending on the case.
Some popular ones are the coefficient of determination $R^2$ and the (root) mean squared error (R)MSE:
\begin{equation}
R^2(\mathbf{y}, \hat{\mathbf{y}}) = 1-\frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2}
\end{equation}

\begin{equation}
\mathrm{MSE}(\mathbf{y}, \hat{\mathbf{y}}) = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2
\end{equation}

\begin{equation}
\mathrm{RMSE}(\mathbf{y}, \hat{\mathbf{y}}) = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2}
\end{equation}

An overview of error metrics can be found here: https://scikit-learn.org/stable/modules/model_evaluation.html#which-scoring-function-should-i-use.

Note that the lower the MSE and RMSE values, the better the model is, while the higher the R2 score (up to 1), the better the model

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error

r2_ols = r2_score(y_test, y_pred_OLS)
mse_ols = mean_squared_error(y_test, y_pred_OLS)
rmse_ols = root_mean_squared_error(y_test, y_pred_OLS)

print(f'R2 for OLS is: {r2_ols:.2f}')
print(f'MSE for OLS is: {mse_ols:.2f}')
print(f'RMSE for OLS is: {rmse_ols:.2f}')


# Lasso regression

In the OLS regression model, we have been relying on all the input features. However, if some features are not correlated with the output this can decrease the accuracy of the model. The LASSO regression model penalizes the coefficients that are different from zero, forcing the weights to be active only if they improve the model.

The objective function of the LASSO regression problem is:

$\mathbf{w} = \underset{\mathbf{w}}{\mathrm{min}} \frac{1}{2 n} ||\mathbf{X}\mathbf{w} - \mathbf{y}||^2_2 + \alpha ||\mathbf{w}||_1$

In which the coefficient $\alpha$ controls how much we regularize the model.

In [ ]:
from sklearn.linear_model import Lasso
Lasso_reg = Lasso(alpha=1.).fit(X0_train, y_train)
y_pred_lasso = Lasso_reg.predict(X0_test)

print('LS coefficients: ')
print(np.round(OLS_reg.coef_, 3))

print('Lasso coefficients: ')
print(np.round(Lasso_reg.coef_, 3))

r2_ols = r2_score(y_test, y_pred_OLS)
r2_lasso = r2_score(y_test, y_pred_lasso)

print(f'R2 for OLS is: {r2_ols:.2f}')
print(f'R2 for Lasso is: {r2_lasso:.2f}')

The penalty on the L1 norm is used to promote the sparsity of the regression weights.

To infer the correct value of $\alpha$ to apply we can use the cross-validation.

### What is cross-validation?

A key and general risk in Machine Learning is overfitting, in which the model learns the training data "too well" (including noise and random fluctuations) rather than the underlying, generalizable patterns. A clear sign of overfitting is a model performing very well on train data, but poorly on the test data. Cross-Validation is a solution to mitigate the risk of overfitting, by traning the model several times on different data splits and averaging its performance.

 In the case of $k$-fold Cross-Validation:

1. Split the data into $k$ equal parts (folds).
2. Train the model $k$ times, each time leaving one fold out as a validation set and using the rest of the data as training data.
3. Average the performance across the $k$ runs.

![Img](https://scikit-learn.org/stable/_images/grid_search_cross_validation.png)



### Using Cross-Validation for our Lasso regression

In our case, we use CV on the training set only, such that we find the optimal $\alpha$ without touching our actual test set. 
In `LassoCV` and `RidgeCV` (which we will see later), scikit-learn uses this procedure to automatically select the best regularization parameter $\alpha$.

In [ ]:
from sklearn.linear_model import LassoCV

LassoCV_reg = LassoCV(cv=5, random_state=42).fit(X0_train, y_train)
y_pred_lassoCV = LassoCV_reg.predict(X0_test)

plt.scatter(y_test, y_pred_lassoCV)
plt.plot(y_test, y_test, c='r', alpha=0.6, ls='--')
plt.xlim(y_test.min()-1, y_test.max()+1)
plt.ylim(y_test.min()-1, y_test.max()+1)
plt.xlabel('Observed target')
plt.ylabel('Predicted target')
plt.show()

print('LS coefficients: ')
print(np.round(OLS_reg.coef_, 3))

print('LassoCV coefficients: ')
print(np.round(LassoCV_reg.coef_, 3))

r2_lassoCV = r2_score(y_test, y_pred_lassoCV)

print(f'R2 for OLS is: {r2_ols:.2f}')
print(f'R2 for LassoCV is: {r2_lassoCV:.2f}')

print(f'alpha = {LassoCV_reg.alpha_:.2f}')

In [ ]:
# Visual comparison of coefficients
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

axes[0].barh(features, OLS_reg.coef_)
axes[0].set_title('OLS coefficients')
axes[0].axvline(0, color='k', lw=0.5)

axes[1].barh(features, Lasso_reg.coef_)
axes[1].set_title('Lasso (α=1) coefficients')
axes[1].axvline(0, color='k', lw=0.5)

axes[2].barh(features, LassoCV_reg.coef_)
axes[2].set_title(f'LassoCV (α={LassoCV_reg.alpha_:.2f}) coefficients')
axes[2].axvline(0, color='k', lw=0.5)

plt.tight_layout()
plt.show()

IMPORTANT: Each coefficient represents a conditional association: it tells us the relationship between that feature and the target while holding all other features constant. This can differ from what you see in the marginal plots from the beginning of the session. Also, regression coefficients do not imply causality. Drawing causal conclusions requires additional assumptions (e.g., experimental design or causal inference methods) that go beyond fitting a regression model.

### Example in which Lasso regression in useful:

Let's add 150 features of pure noise and compare how a simple Ordinary Least-Square Regression and a Lasso regression would handle it.

In [ ]:
# Add 150 columns of pure noise

np.random.seed(42)
n_noise = 150
X_noise = np.random.randn(X.shape[0], n_noise)
X_aug = np.hstack([X, X_noise])
features_aug = features + [f'noise_{i}' for i in range(n_noise)]

print(f'Original number of features: {X.shape[1]}')
print(f'After adding noise: {X_aug.shape[1]}')

In [ ]:
# Train/test split and scaling (same procedure as before)
X_tr, X_te, y_tr, y_te = train_test_split(X_aug, y, test_size=0.2, random_state=1)

scaler_aug = StandardScaler()
X_tr_s = scaler_aug.fit_transform(X_tr)
X_te_s = scaler_aug.transform(X_te)

# OLS
ols_aug = LinearRegression().fit(X_tr_s, y_tr)
y_pred_ols_aug = ols_aug.predict(X_te_s)

# LassoCV
lasso_aug = LassoCV(cv=5, random_state=42).fit(X_tr_s, y_tr)
y_pred_lasso_aug = lasso_aug.predict(X_te_s)

print(f'R² OLS:     {r2_score(y_te, y_pred_ols_aug):.3f}')
print(f'R² LassoCV: {r2_score(y_te, y_pred_lasso_aug):.3f}  (α = {lasso_aug.alpha_:.2f})')
print(lasso_aug.coef_)

As expected, we can see that the Lasso regression handles much better irrelevant features. The penalty on the L1 norm forces the weight associated to a feature to be zero if irrelevant.

## Ridge regression

In other cases, you could have corelated variables, which are all equally important. You would still want to use this data for the regression, while still wanting robustness against noise. Ridge regression allows to do this, as the regularization is applied to the $l_2$ norm of the weights. This allows to reduce the total magnitude of the weights, so that the model is less sensitive to noise, while still keeping all features for the regression.
The objective function of a Ridge regression problem is:

$\mathbf{w} = \underset{\mathbf{w}}{\mathrm{min}} ||\mathbf{X}\mathbf{w} - \mathbf{y}||^2_2 + \alpha ||\mathbf{w}||_2$

In [ ]:
from sklearn.linear_model import RidgeCV

RidgeCV_reg = RidgeCV(alphas=(0.1, 0.5, 1, 5, 10, 50), cv=5).fit(X0_train, y_train)
y_pred_RidgeCV = RidgeCV_reg.predict(X0_test)

plt.scatter(y_test, y_pred_RidgeCV)
plt.plot(y_test, y_test, c='r', alpha=0.6, ls='--')
plt.xlim(y_test.min()-1, y_test.max()+1)
plt.ylim(y_test.min()-1, y_test.max()+1)
plt.xlabel('Observed target')
plt.ylabel('Predicted target')
plt.show()

print('OLS coefficients: ')
print(np.round(OLS_reg.coef_, 3))

print('RidgeCV coefficients: ')
print(np.round(RidgeCV_reg.coef_, 3))

r2_ridgeCV = r2_score(y_test, y_pred_RidgeCV)

print(f'R2 for OLS is: {r2_ols:.2f}')
print(f'R2 for RidgeCV is: {r2_ridgeCV:.2f}')

print(f'alpha = {RidgeCV_reg.alpha_:.2f}')

Let's illustrate a case in which the Ridge regression can be useful. The following dataset is created artificially: instead of adding pure noise (like we did to illustrate the use of Lasso), we add new features, which are correlated to the initial ones. This is purely for didactical purposes: we assume that these features represent something and are just as important as the other features

In [ ]:
X_aug = None
features_aug = None
np.random.seed(42)
n_decoys = 250
noise_level = 0.1

decoy_features = []
for i in range(n_decoys):
    # Pick 3 random real columns
    cols = np.random.choice(X.shape[1], size=3, replace=False)
    
    # Random convex weights: x, y, 1-x-y (with x+y <= 1)
    x = np.random.uniform(0, 1)
    y_w = np.random.uniform(0, 1 - x)
    z_w = 1 - x - y_w
    
    decoy = (x * X[:, cols[0]] 
             + y_w * X[:, cols[1]] 
             + z_w * X[:, cols[2]] 
             + noise_level * np.random.randn(X.shape[0]))
    
    decoy_features.append(decoy)

X_aug = np.hstack([X, np.column_stack(decoy_features)])
features_aug = features + [f'decoy_{i}' for i in range(n_decoys)]

print(f'Original features: {X.shape[1]}')
print(f'After adding decoys: {X_aug.shape[1]}')

In [ ]:
# Train/test split and scaling (same procedure as before)
X_tr, X_te, y_tr, y_te = train_test_split(X_aug, y, test_size=0.2, random_state=1)

scaler_aug = StandardScaler()
X_tr_s = scaler_aug.fit_transform(X_tr)
X_te_s = scaler_aug.transform(X_te)

# OLS
ols_aug = LinearRegression().fit(X_tr_s, y_tr)
y_pred_ols_aug = ols_aug.predict(X_te_s)

# LassoCV, just for fun
lasso_aug = LassoCV(cv=5, random_state=42).fit(X_tr_s, y_tr)
y_pred_lasso_aug = lasso_aug.predict(X_te_s)

# RidgeCV
ridge_aug = RidgeCV(alphas=(0.1, 0.5, 1, 5, 10, 50),cv=5).fit(X_tr_s, y_tr)
y_pred_ridge_aug = ridge_aug.predict(X_te_s)



print(f'R² OLS:     {r2_score(y_te, y_pred_ols_aug):.3f}')
print(f'R² LassoCV: {r2_score(y_te, y_pred_lasso_aug):.3f}  (α = {lasso_aug.alpha_:.2f})')
print(f'R² RidgeCV: {r2_score(y_te, y_pred_ridge_aug):.3f}  (α = {ridge_aug.alpha_:.2f})')
print(ridge_aug.coef_)
print(ols_aug.coef_)

We see that Ridge regression accounts for all variables, but keeps much lower values for the weights associated to each variable. This is as expected, given the ridge regression formula. We also see Lasso performing well, but since all features are important in this case, we are happier with Ridge. By printing the Lasso coefficients, you can observe that Lasso forced many weights to be zero.

## Principal components regression

The principal component regression is the same as the OLS regression, with an extra-step: the PCA is applied to the X matrix, and the linear regression is performed on the new projected data.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(X0_train)

plt.scatter(np.arange(X.shape[1]), pca.explained_variance_ratio_*100)
plt.xlabel('PCs')
plt.ylabel('Explained variance')
plt.show()

In [ ]:
A_train = pca.components_.T
Z_train = X0_train @ A_train
Z_test = X0_test @ A_train

# The regressio has to be applied to the PC scores
PCR_reg = LinearRegression().fit(Z_train, y_train)
y_pred_PCR = PCR_reg.predict(Z_test)

plt.scatter(y_test, y_pred_PCR)
plt.plot(y_test, y_test, c='r', alpha=0.6, ls='--')
plt.xlim(y_test.min()-1, y_test.max()+1)
plt.ylim(y_test.min()-1, y_test.max()+1)
plt.xlabel('Observed target')
plt.ylabel('Predicted target')
plt.show()

print('LS coefficients: ')
print(np.round(OLS_reg.coef_, 3))

print('PCR coefficients: ')
print(np.round(PCR_reg.coef_, 3))

r2_pcr = r2_score(y_test, y_pred_PCR)

print(f'R2 for OLS is: {r2_ols:.2f}')
print(f'R2 for PCR is: {r2_pcr:.2f}')

This added step has two benefits:

* The features become uncorrelated between them.
* The dimensionality of the feature matrix can be reduced.

In [ ]:
# We can test the regression with fewer features
q = 5
Z_train = X0_train @ A_train[:,:q]
Z_test = X0_test @ A_train[:,:q]

PCR_reg = LinearRegression().fit(Z_train, y_train)
y_pred_PCR = PCR_reg.predict(Z_test)

plt.scatter(y_test, y_pred_PCR)
plt.plot(y_test, y_test, c='r', alpha=0.6, ls='--')
plt.xlim(y_test.min()-1, y_test.max()+1)
plt.ylim(y_test.min()-1, y_test.max()+1)
plt.xlabel('Observed target')
plt.ylabel('Predicted target')
plt.show()

r2_pcr = r2_score(y_test, y_pred_PCR)

print(f'R2 for OLS is: {r2_ols:.2f}')
print(f'R2 for PCR is: {r2_pcr:.2f}')

## Summary: when to use which method?

| Method | Use when... | Trade-off |
|--------|------------|----------|
| **OLS** | You have few features and trust they are all relevant. | No regularization — sensitive to noise and multicollinearity. |
| **Lasso** | You suspect many features are irrelevant and want automatic feature selection. | L1 penalty drives some coefficients exactly to zero. |
| **Ridge** | You expect many features to have small but nonzero contributions and want to reduce sensitivity to noise. | L2 penalty shrinks all coefficients but never zeroes them out. |
| **PCR** | Features are highly correlated and you want to work in a decorrelated, lower-dimensional space. | PCs that explain the most variance in X may not be the most predictive of y — consider PLS as an alternative. |

In [ ]:
# ── Summary comparison of all methods ──
from sklearn.metrics import r2_score, root_mean_squared_error

methods = ['OLS', 'Lasso (α=1)', 'LassoCV', 'RidgeCV', 'PCR (all PCs)', 'PCR (q=5)']

# Recompute PCR with all PCs for the summary
Z_train_all = X0_train @ A_train
Z_test_all = X0_test @ A_train
PCR_all = LinearRegression().fit(Z_train_all, y_train)
y_pred_PCR_all = PCR_all.predict(Z_test_all)

# PCR with q=5 is already the last one computed
predictions = [y_pred_OLS, y_pred_lasso, y_pred_lassoCV, y_pred_RidgeCV, y_pred_PCR_all, y_pred_PCR]

summary = pd.DataFrame({
    'Method': methods,
    'R²': [r2_score(y_test, yp) for yp in predictions],
    'RMSE': [root_mean_squared_error(y_test, yp) for yp in predictions]
}).round(3)

print(summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
axes[0].barh(summary['Method'], summary['R²'], color=colors)
axes[0].set_xlabel('R²')
axes[0].set_title('R² (higher is better)')

axes[1].barh(summary['Method'], summary['RMSE'], color=colors)
axes[1].set_xlabel('RMSE')
axes[1].set_title('RMSE (lower is better)')

plt.tight_layout()
plt.show()